Script Overview:
1. Excluded words are combined with spaCys default stop words
2. Docs are tokenized from line 6 (excluding date and title) and words are excluded
3. Words with a frequency of one or two are removed and terms occuring in more than 95% of the docs are removed
4. The tokenized text is prepared for topic modeling and converted to a BOW
5. Hyperparameter optimization is performed to find the best parameters for the LDA model
6. LDA model is run with best parameters
7. Topics are visualized 
8. Top 10 words per topic are printed
9. Uncertainty terms calculated per doc
10. Number of docs per topic are calculated through the probability distribution given for each document from LDA (e.g if 50% of doc 1 is topic 2 then topic 2 gets given 0.5)
11. Number of uncertainty terms per topic is calculated the same way (e.g. doc 1 has 5 uncertainty terms, 50% of doc 1 is topic 2 so topic 2 gets assigned 2.5 words)
12. Chi square goodness of fit assumption check, if met test is performed if not permutation test is performed. 

1. Excluded words are combined with spaCys default stop words

In [2]:
excluded_words = ["the", "in", "to", "as","an", "of", "and", "from", "a", "by", "for", "their", "than", "were", "this", "or", "is", 
                  "which","was", "have", "with","  ","those", "who", "on", "had", "that", "but", "into", "are", "also", "$", "\n",
                    "\xa0","toll-free1-800-263-1136;514-283-8300;infostats@statcan.gc.ca", "article",
                    "statcan.mediahotline-ligneinfomedias.statcan@statcan.gc.ca", "statcan", "ca", "canada", "data",
                    "gc", "canadian", "among", "billion", "at", "year", "released", "release", "about", "reported", "be", "increase",
                    "report", "high", "rate", "age", "people", "over", "available", "million", "non", "all", 
                    "population", "quarter", "total", "low", "large", "rise", "decline", "use", "level"]
excluded_words.extend(nlp.Defaults.stop_words)
excluded_words = list(set(excluded_words))


2. Docs are tokenized from line 6 (excluding date and title) and words are excluded

In [3]:

import spacy 
import os
#pip install en_core_web_sm-3.8.0-py3-none-any.whl
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# first script run through
import re
nlp = spacy.load("en_core_web_sm")


# create a function to tokenize text 
excluded_tokens = set()
def tokenize_text(file):
    with open(file, "r") as f:
        lines = f.readlines()[6:] # from text content
        
    text = "".join(lines)

    # create doc with tokens
    doc = nlp(text)

    # remove punctuation and excluded words
    words = []
    for token in doc:
        if token.is_alpha and token.lemma_.lower() not in excluded_words: 
            words.append(token.lemma_.lower())
        else:
            excluded_tokens.add(token.text.lower())

    return " ".join(words)


# create a function to loop over all the articles in the directory 
directory = "scraped_a"

docs = []
filenames = []
for file in os.listdir(directory):
    file_path = os.path.join(directory, file)
    token_text = tokenize_text(file_path)
    docs.append(token_text)
    filenames.append(file)

vectorizer = CountVectorizer(stop_words = excluded_words)

# outputs a sparse matrix
Canada_matrix = vectorizer.fit_transform(docs)



/home/iwag@cbsp.nl/.local/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['1136', '263', '283', '514', '800', '8300', 'free1', 'infostats', 'ligneinfomedias', 'll', 'mediahotline', 'toll', 've'] not in stop_words.
  warnings.warn(


3. Words with a frequency of one or two are removed and terms occuring in more than 95% of the docs are removed

In [4]:
# convert to a dataframe 
canada_df = pd.DataFrame(Canada_matrix.toarray(), index = filenames, columns = vectorizer.get_feature_names_out())

# remove cols with one or two terms
c_totals_1 = canada_df.columns[canada_df.sum(axis = 0)==1]
canada_df = canada_df.drop(columns = c_totals_1)
c_totals_2 = canada_df.columns[canada_df.sum(axis = 0)==2]
canada_df = canada_df.drop(columns = c_totals_2)
term_counts = (canada_df > 0).sum(axis = 0)

# getting terms that occur in more than 95% of docs
terms_95 = canada_df.columns[term_counts > 0.95*len(canada_df)]

canada_df = canada_df.drop(columns= terms_95)


In [5]:
total = canada_df.sum()
t = total.sort_values(ascending = False)
t.head(10)

compare       873
woman         819
service       800
business      773
child         762
industry      656
income        654
group         640
health        630
indigenous    624
dtype: int64

4. The tokenized text is prepared for topic modeling and converted to a BOW

In [6]:
## converting the matrix into bag of words format for LDA

import gensim
from scipy.sparse import csr_matrix
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [7]:
dictionary = Dictionary([[word] for word in canada_df.columns])
bow = []
for idx, row in canada_df.iterrows():
    doc_bow = [(dictionary.token2id[word], count) for word, count in row.items() if count > 0]
    bow.append(doc_bow)


5. Hyperparameter optimization is performed to find the best parameters for the LDA model

In [ ]:
# optimize hyperparameters
import numpy as np

num_topics_range = [20,25,28,30,31]
alpha =["auto","symmetric", "asymmetric", 0.1,0.3,0.5, 0.8]
eta = ["auto","symmetric", 0.01, 0.03, 0.05, 0.08, 0.1, 0.5, 0.7, 0.9]


def coherence_score_fnc(model, texts, dictionary):
    cv_model = CoherenceModel(model = model, texts = texts, dictionary = dictionary, coherence= "c_v")
    cv_score = cv_model.get_coherence()
    return cv_score

texts = [doc.split() for doc in docs]

best_coherence = -1
best_model = None
best_params = None


# set a seed for reproducibility
seed = 42

scores = []

for num_topics in num_topics_range:
    for a in alpha:
        for e in eta:
            print(f"Model with topics:{num_topics}, alpha:{a}, eta:{e}")

            lda_model = gensim.models.LdaModel(bow, num_topics= num_topics, id2word = dictionary, passes= 10, alpha = a, eta = e, random_state= seed)

            coherence_score = coherence_score_fnc(model = lda_model, texts = texts, dictionary= dictionary)
            print(f"Coherence Score: {coherence_score}")

            scores.append((coherence_score, num_topics, a ,e))

        

scores.sort(reverse = True, key = lambda x:x[0])
for i in range(min(5, len(scores))):
    print(f"Position {i + 1}: Coherence Score = {scores[i][0]}, Parameters: num_topics = {scores[i][1]}, alpha = {scores[i][2]}, eta = {scores[i][3]}")




Model with topics:20, alpha:auto, eta:auto
Coherence Score: 0.4199363809028774
Model with topics:20, alpha:auto, eta:symmetric
Coherence Score: 0.4199363809028774
Model with topics:20, alpha:auto, eta:0.01
Coherence Score: 0.4274544260985868
Model with topics:20, alpha:auto, eta:0.03
Coherence Score: 0.42342567638589035
Model with topics:20, alpha:auto, eta:0.05
Coherence Score: 0.4199363809028774
Model with topics:20, alpha:auto, eta:0.08
Coherence Score: 0.4214867734093983
Model with topics:20, alpha:auto, eta:0.1
Coherence Score: 0.42243161787121936
Model with topics:20, alpha:auto, eta:0.5
Coherence Score: 0.41741739829596486
Model with topics:20, alpha:auto, eta:0.7
Coherence Score: 0.451550709279961
Model with topics:20, alpha:auto, eta:0.9
Coherence Score: 0.44985369924451124
Model with topics:20, alpha:symmetric, eta:auto
Coherence Score: 0.42045811741545247
Model with topics:20, alpha:symmetric, eta:symmetric
Coherence Score: 0.42045811741545247
Model with topics:20, alpha:sym

6. LDA model is run with best parameters

In [8]:
import numpy as np
# best results show asymmetric alpha = 0.1, 0.9 eta and 20 topics 
lda_gensim1 = gensim.models.LdaModel(
    bow, num_topics=20, id2word=dictionary, passes=10, alpha= 0.1, eta=0.9, random_state= 42
)

coherence_model = CoherenceModel(model= lda_gensim1, texts = [doc.split() for doc in docs] , dictionary= dictionary,
                              coherence = "c_v")
with np.errstate(invalid= "ignore"): 
    c_cv = coherence_model.get_coherence()  

#print measure
c_cv 
    

0.4536443251516615

7. Topics are visualized 

In [9]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
visual = gensimvis.prepare(lda_gensim1, bow, dictionary)
pyLDAvis.display(visual)

8. Top 10 words per topic are printed

In [10]:
# Extract top 5 words and their weights for each topic
for i, topic in enumerate(lda_gensim1.show_topics(num_topics=20, num_words=5, formatted=False)):
    print(f"Topic {i+1}:")
    for word, weight in topic[1]:
        print(f"  {word}: {weight}")

Topic 1:
  cannabis: 0.019074948504567146
  membership: 0.006986354943364859
  plan: 0.006465590558946133
  digital: 0.0062155104242265224
  witness: 0.0047437893226742744
Topic 2:
  incident: 0.02442743629217148
  crime: 0.02408786676824093
  police: 0.021071575582027435
  victim: 0.020406734198331833
  homicide: 0.017193889245390892
Topic 3:
  passenger: 0.008192690089344978
  ptsd: 0.006836180575191975
  movement: 0.006579705514013767
  event: 0.006403082050383091
  december: 0.005783567205071449
Topic 4:
  fatality: 0.014911230653524399
  vehicle: 0.01265326514840126
  motorcycle: 0.011427637189626694
  death: 0.008845520205795765
  driver: 0.00860864669084549
Topic 5:
  revenue: 0.03179722651839256
  industry: 0.024532388895750046
  operating: 0.018461741507053375
  sale: 0.01441084686666727
  expense: 0.012812994420528412
Topic 6:
  income: 0.01542529184371233
  executive: 0.014274164102971554
  average: 0.011836676858365536
  tax: 0.010387145914137363
  filer: 0.0101954638957977

9. Uncertainty terms calculated per doc

In [11]:

# first lets start by tokenizing the text and looking for uncertainty 
import os
import spacy
from nltk.util import ngrams
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



nlp = spacy.load("en_core_web_sm")



uncertainty_terms = set([
    "always", "certain", "chance", "common", "doubtful", "expected","frequently","generally", "likely", 
    "impossible","never","inconclusive","often",
    "possible", "predicatable", "probable","rarely", "seldom", "slightly", "slight", "sometimes", 
    "uncertain", "uncommon","usually", "unlikely"
])


# tokenization fnc
excluded_tokens = set()
def tokenize_text(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()[7:]
        text = "".join(lines)

    doc = nlp(text)
    return doc 
# uncertainty fnc
def find_uncertainty_terms(doc_tokens):
    found_terms = []
    found_terms += [w.text.lower() for w in doc_tokens if w.text.lower() in uncertainty_terms]
    return found_terms


# Directory with files
directory = "scraped_a"  


tokenize_docs = []
filenames = []

total_u_counts = 0

for file in os.listdir(directory):
    file_path = os.path.join(directory, file)
    doc = tokenize_text(file_path)
    filenames.append(file)
    tokenize_docs.append(doc)
    

10. Number of docs per topic are calculated through the probability distribution given for each document from LDA (e.g if 50% of doc 1 is topic 2 then topic 2 gets given 0.5)
11. Number of uncertainty terms per topic is calculated the same way (e.g. doc 1 has 5 uncertainty terms, 50% of doc 1 is topic 2 so topic 2 gets assigned 2.5 words)

In [12]:
# Initialize topic uncertainty counts with weights
topic_u_weighted_counts = {i:0 for i in range(lda_gensim1.num_topics)}
topic_doc_counts = {i:0 for i in range(lda_gensim1.num_topics)}

for idx, content in enumerate(bow):
    doc_topic_dist = lda_gensim1.get_document_topics(content)
    doc_tok = tokenize_docs[idx]
 
    found_terms = find_uncertainty_terms(doc_tok)
    num_uncertainty_terms = len(found_terms)
    
    
    for topic, prob in doc_topic_dist:
        topic_doc_counts[topic] += prob
        weighted_uncertainty = num_uncertainty_terms * prob
        topic_u_weighted_counts[topic] += weighted_uncertainty


topic_u_weighted_counts

{0: 15.755547917447984,
 1: 119.57185269892216,
 2: 9.975759878754616,
 3: 29.604495983570814,
 4: 33.884071389213204,
 5: 18.099336981773376,
 6: 133.863491890952,
 7: 22.152154486626387,
 8: 11.615384489297867,
 9: 342.0916763348505,
 10: 49.0773339048028,
 11: 19.554485984146595,
 12: 78.97443238645792,
 13: 138.31091360002756,
 14: 88.21802299842238,
 15: 16.67179437726736,
 16: 7.963767744600773,
 17: 94.0398117415607,
 18: 31.33984594605863,
 19: 108.23961048666388}

In [13]:
data = {"Topic": list(topic_u_weighted_counts.keys()),
        "Uncertainty Terms": list(topic_u_weighted_counts.values()),
        "Documents": list(topic_doc_counts.values())
        }


df =pd.DataFrame(data)
df["Relative Freq"] = df["Uncertainty Terms"]/df["Documents"]

12. Chi square goodness of fit assumption check, if met test is performed if not permutation test is performed. 

In [ ]:
# we see E < 5 , Chi Square cannot be performed
obs = df["Relative Freq"]
exp = [sum(obs)/len(obs)]*len(obs)
exp

[2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588,
 2.047225206901588]

In [16]:
# since chi square assumption not met we do the permutation test 

obs_values = df["Relative Freq"]
obs_var = np.var(obs_values)

n_perm = 10000
perm_vars = []

for i in range(n_perm):
    shuff_values = np.random.permutation(obs_values)
    perm_var = np.var(shuff_values)
    perm_vars.append(perm_var)


p_value = np.sum(np.array(perm_vars)>= obs_var)/n_perm

p_value

0.9937

In [17]:
obs_var

2.16038485017753